# Kaggle: Download Audio Zip from Google Drive

**Single 5GB zip file** - much faster than 621 individual files!

Steps:
1. Download zip from Drive (10-15 min)
2. Extract audio (5 min)
3. Extract features (30 min)
4. Train model

**Total time: ~45-60 minutes**

In [ ]:
# === STEP 1: INSTALL DEPENDENCIES ===
!pip install -q gdown librosa numpy pandas scikit-learn tqdm

import os
import subprocess

OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Dependencies installed!')

In [ ]:
# === STEP 2: DOWNLOAD ZIP FROM GOOGLE DRIVE ===
# REPLACE WITH YOUR ZIP FILE'S GOOGLE DRIVE LINK
# 
# After uploading the zip to Drive, right-click it → Share → Anyone with link
# Copy the link and extract the FILE ID
#
# URL format: https://drive.google.com/file/d/FILE_ID/view
# FILE_ID is the part between /d/ and /view

ZIP_FILE_ID = 'YOUR_ZIP_FILE_ID_HERE'  # REPLACE THIS

print(f'Downloading zip file...')
!gdown {ZIP_FILE_ID} -O {OUTPUT_DIR}/audio.zip

print('Download complete!')

In [ ]:
# === STEP 3: EXTRACT ZIP ===
import zipfile

zip_path = f'{OUTPUT_DIR}/audio.zip'
extract_dir = f'{OUTPUT_DIR}/audio'
os.makedirs(extract_dir, exist_ok=True)

print('Extracting zip...')
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print('Extraction complete!')

# Find audio files
audio_files = []
for root, dirs, files in os.walk(extract_dir):
    for f in files:
        if f.endswith('.m4a'):
            audio_files.append(os.path.join(root, f))

print(f'Found {len(audio_files)} audio files')

In [ ]:
# === STEP 4: EXTRACT FEATURES ===
import numpy as np
import librosa
from tqdm import tqdm

def extract_features(audio_path, sr=22050):
    try:
        y, _ = librosa.load(audio_path, sr=sr, mono=True)
        
        hop_length = 512
        rms = librosa.feature.rms(y=y, hop_length=hop_length)[0]
        times = librosa.times_like(rms, sr=sr, hop_length=hop_length)
        silence_mask = rms < 0.01
        
        utterances = []
        in_utt = False
        
        for i, (t, is_sil) in enumerate(zip(times, silence_mask)):
            if not is_sil and not in_utt:
                start_idx = i
                in_utt = True
            elif is_sil and in_utt:
                end_idx = i
                start_t = times[start_idx]
                end_t = times[end_idx]
                
                if end_t - start_t > 0.3:
                    rms_vals = rms[start_idx:end_idx]
                    
                    start_sample = int(start_t * sr)
                    end_sample = int(end_t * sr)
                    segment = y[start_sample:end_sample]
                    
                    if len(segment) < 1024:
                        in_utt = False
                        continue
                    
                    feat = []
                    
                    # Energy
                    feat.extend([
                        np.mean(rms_vals),
                        np.max(rms_vals),
                        np.std(rms_vals),
                        np.max(rms_vals) / (np.mean(rms_vals) + 1e-8)
                    ])
                    
                    # ZCR
                    zcr = librosa.feature.zero_crossing_rate(segment)[0]
                    feat.extend([np.mean(zcr), np.std(zcr)])
                    
                    # MFCCs
                    mfccs = librosa.feature.mfcc(y=segment, sr=sr, n_mfcc=13)
                    for i in range(13):
                        feat.extend([np.mean(mfccs[i]), np.std(mfccs[i])])
                    
                    # Spectral
                    spec_cent = librosa.feature.spectral_centroid(y=segment, sr=sr)[0]
                    spec_bw = librosa.feature.spectral_bandwidth(y=segment, sr=sr)[0]
                    spec_rolloff = librosa.feature.spectral_rolloff(y=segment, sr=sr)[0]
                    feat.extend([np.mean(spec_cent), np.std(spec_cent)])
                    feat.extend([np.mean(spec_bw), np.std(spec_bw)])
                    feat.extend([np.mean(spec_rolloff), np.std(spec_rolloff)])
                    
                    label = 1 if np.max(rms_vals) > 0.15 else 0
                    
                    utterances.append({
                        'features': np.array(feat, dtype=np.float32),
                        'label': label
                    })
                
                in_utt = False
        
        return utterances
    except:
        return []

print('Extracting features...')
all_samples = []

for audio_file in tqdm(audio_files[:100]):  # Start with 100
    samples = extract_features(audio_file)
    all_samples.extend(samples)

print(f'Processed {len(audio_files[:100])} files, {len(all_samples)} utterances')

In [ ]:
# === STEP 5: TRAIN MODEL ===
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
import pickle

X = np.array([s['features'] for s in all_samples])
y = np.array([s['label'] for s in all_samples])

print(f'Dataset: {len(X)} samples, pos={y.sum()} ({100*y.mean():.1f}%)')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X_train_s, y_train)

y_pred = clf.predict(X_test_s)
f1 = f1_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)

print(f'\n=== RESULTS ===')
print(f'F1: {f1:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Reacall: {rec:.4f}')

# Save
with open(f'{OUTPUT_DIR}/model.pkl', 'wb') as f:
    pickle.dump({'model': clf, 'scaler': scaler, 'f1': f1}, f)
print('\nModel saved!')